# Share five weeks in a sheet

*Data Analysis in Python · Topic 12.5*

Your team wants November and December 2025's weekly sessions, transactions and revenue in a Google Sheet. Write the table to a sheet of your own, and read it back to check it.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the cell that creates the `weekly` table, then the login cell. Choose the Google account you use for Colab, and **Allow**.
2. In the first cell marked ✏️, create `spreadsheet`: the spreadsheet called `SHEET_NAME`, opened if it exists and created if it doesn't, and `worksheet`: its first tab.
3. In the second cell marked ✏️, empty the worksheet, then write the `weekly` table to it: a header row with the column names, then one row per week. Write `week_start` as text, such as `2025-11-03`.
4. In the last cell marked ✏️, create the `read_back` table: every row of the worksheet, read back from Google Sheets.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. Then open `spreadsheet.url` and look at your sheet.

**Expected output**: A spreadsheet called "Simmer weekly report" in your Google Drive. The `read_back` table has the 5 weeks, from 2025-11-03 to 2025-12-01, with 222,145 sessions, 7,800 transactions and $605,905.28 of revenue.

## Parameters

In [ ]:
import pandas as pd

SHEET_NAME = "Simmer weekly report"

# Five weeks of November and December 2025. Run it, and leave it as it is.
weekly = pd.DataFrame({
    "week_start": pd.to_datetime(["2025-11-03", "2025-11-10", "2025-11-17", "2025-11-24", "2025-12-01"]),
    "sessions": [33577, 35686, 41407, 64194, 47281],
    "transactions": [738, 914, 1701, 2640, 1807],
    "revenue": [68719.10, 86301.97, 133375.37, 171097.72, 146411.12],
})
weekly

In [ ]:
# Log in to Google as yourself. Run it, choose your account and Allow.
import google.auth
import gspread
from google.colab import auth

auth.authenticate_user()
credentials, _ = google.auth.default()
gc = gspread.authorize(credentials)

## The spreadsheet

In [ ]:
# ✏️ Create spreadsheet and worksheet


## The table

In [ ]:
# ✏️ Write the weekly table to the worksheet


## Read it back

In [ ]:
# ✏️ Create read_back


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None and "JSON serializable" in str(error):
            results.append((None, f"A cell stopped with {type(error).__name__}: {error}. Turn week_start into text with .dt.strftime(\"%Y-%m-%d\") before you write it, then choose Runtime → Restart session and run all."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    spreadsheet = found.get("spreadsheet")
    if spreadsheet is None or not hasattr(spreadsheet, "sheet1"):
        results.append((False, "spreadsheet should be a gspread spreadsheet: gc.open(SHEET_NAME), or gc.create(SHEET_NAME) if it doesn't exist."))
    elif spreadsheet.title != found.get("SHEET_NAME"):
        results.append((False, f"The spreadsheet should be called {found.get('SHEET_NAME')!r}. It's called {spreadsheet.title!r}."))
    else:
        results.append((True, f"The spreadsheet {spreadsheet.title!r} is in your Google Drive."))

    read_back = found.get("read_back")
    columns = ["week_start", "sessions", "transactions", "revenue"]
    if not isinstance(read_back, pd.DataFrame):
        results.append((False, "read_back should be a DataFrame: pd.DataFrame(worksheet.get_all_records())."))
    elif list(read_back.columns) != columns:
        results.append((False, f"The sheet's header row should be the weekly table's columns, {columns}. read_back has {list(read_back.columns)}. Write [weekly.columns.tolist()] first, then the rows."))
    elif len(read_back) != 5:
        results.append((False, f"The sheet should have 5 weeks below its header. read_back has {len(read_back)} rows. Clear the worksheet before you write."))
    elif [str(week) for week in read_back["week_start"]] != ["2025-11-03", "2025-11-10", "2025-11-17", "2025-11-24", "2025-12-01"]:
        results.append((False, f"week_start should be written as text, such as 2025-11-03. The sheet has {list(read_back['week_start'])[:2]}."))
    elif int(read_back["sessions"].sum()) != 222145 or int(read_back["transactions"].sum()) != 7800 or abs(read_back["revenue"].sum() - 605905.28) > 0.01:
        results.append((False, f"The sheet should have 222,145 sessions, 7,800 transactions and $605,905.28 of revenue, as the weekly table does. It has {read_back['sessions'].sum():,}, {read_back['transactions'].sum():,} and ${read_back['revenue'].sum():,.2f}."))
    else:
        results.append((True, "The sheet reads back as the weekly table: 5 weeks, 222,145 sessions and $605,905.28 of revenue."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Open spreadsheet.url to see your sheet.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
try:
    spreadsheet = gc.open(SHEET_NAME)
except gspread.SpreadsheetNotFound:
    spreadsheet = gc.create(SHEET_NAME)
worksheet = spreadsheet.sheet1
spreadsheet.url
```

```python
weekly["week_start"] = weekly["week_start"].dt.strftime("%Y-%m-%d")
worksheet.clear()
worksheet.update([weekly.columns.tolist()] + weekly.values.tolist())
```

```python
read_back = pd.DataFrame(worksheet.get_all_records())
read_back
```

</details>